# TorGen: DETR-CVAE for Tornado Outbreak Generation

Install the package, mount Drive, and train.

In [ ]:
!pip install -q git+https://github.com/jcaramichaellehigh/TorGen.git
!pip show torgen | grep -E "^(Version|Location)"
!python -c "
from torgen.loss.hungarian import _focal_bce, HungarianMatchingLoss
from torgen.data.dataset import coords_to_bearing_length
from torgen.model.decoder import TrackDecoder
from torgen.model.track_encoder import PosteriorTrackEncoder
from torgen.eval.evaluate import run_evaluation
import inspect

# v2: z-only decoder (no spatial_tokens / cross-attention)
src = inspect.getsource(TrackDecoder.forward)
assert 'z_broadcast' in src, 'MISSING: z injection'
assert 'spatial_tokens' not in inspect.signature(TrackDecoder.forward).parameters, 'v2 decoder should not take spatial_tokens'

# v2: separate output heads
import torch
dec = TrackDecoder(num_queries=4, d_model=32, d_latent=8, n_layers=1, n_heads=2)
out = dec(torch.randn(1, 8))
assert 'bearing' in out and 'length' in out, 'MISSING: separate bearing/length heads'
assert out['coords'].shape[-1] == 2, f'coords should be 2-dim, got {out[\"coords\"].shape[-1]}'

# v2: simplified track encoder (no transformer layers)
assert 'n_layers' not in inspect.signature(PosteriorTrackEncoder.__init__).parameters, 'v2 track encoder should not have n_layers'

# v2: config defaults
from torgen.training.config import TrainConfig
cfg = TrainConfig(drive_dir='', checkpoint_dir='')
assert cfg.lambda_bearing == 2.0, f'lambda_bearing={cfg.lambda_bearing}'
assert cfg.lambda_length == 2.0, f'lambda_length={cfg.lambda_length}'
assert cfg.n_decoder_layers == 4, f'n_decoder_layers={cfg.n_decoder_layers}, expected 4'
assert not hasattr(cfg, 'memory_dropout'), 'v2 should not have memory_dropout'
assert not hasattr(cfg, 'n_posterior_layers'), 'v2 should not have n_posterior_layers'

print('v2 architecture verified: z-only decoder, separate heads, mean-pool track encoder, updated config')
"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Optional: experiment tracking
# Sign up at wandb.ai, get API key from wandb.ai/authorize
try:
    import wandb
    wandb.login()
except ImportError:
    print("wandb not installed, skipping experiment tracking")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from torgen.training import TrainConfig, train

config = TrainConfig(
    drive_dir="/content/drive/MyDrive/raw",
    checkpoint_dir="/content/drive/MyDrive/checkpoints",
)

trainer = train(config)

In [ ]:
print(f"Epochs trained: {trainer.epoch}")
print(f"Final train loss: {trainer.train_losses[-1]:.4f}")
print(f"Final val loss: {trainer.val_losses[-1]:.4f}")
print(f"Best val loss: {trainer.best_val_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(trainer.train_losses) + 1), trainer.train_losses, label="Train")
ax.plot(range(1, len(trainer.val_losses) + 1), trainer.val_losses, label="Val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("TorGen Training Curves")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
import json
import os

eval_dir = os.path.join(config.checkpoint_dir, "eval")
summary_path = os.path.join(eval_dir, "summary.json")
if os.path.exists(summary_path):
    with open(summary_path) as f:
        results = json.load(f)
    print("Evaluation Results:")
    for k, v in results.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
else:
    print("No evaluation results found (evaluation runs automatically after training)")